# CURE-Rec — final required experiments

Run all cells. This notebook executes the remaining code-backed experiments and writes auditable manifests. It does not fabricate real intervention evidence. The divergent-selector experiment uses disclosed controlled regimes where masks differ; the real-intervention validation and integrated 8/10 policy study are reported as blocked unless their required data/operators are available.


In [1]:
from pathlib import Path
import sys,json
import pandas as pd

C=[Path.cwd(),Path.cwd()/'paper-ideas'/'CURE-Rec'/'code',*Path.cwd().parents]
ROOT=next(p for p in C if (p/'pyproject.toml').exists() and (p/'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from cure_rec.config import load_settings
from cure_rec.observability import RunLogger
from cure_rec.regimes import run_regime_suite

CONFIG=ROOT/'configs'/'curesim_quickstart.yaml'
RESULTS=ROOT/'results'/'reviewer_phase_assets'/'final_required'
RESULTS.mkdir(parents=True,exist_ok=True)
RUN_ALL=True
RUN_DIVERGENT_SELECTOR=True
RUN_REAL_INTERVENTION=False
RUN_INTEGRATED_SCALING=False
print('Root:',ROOT)

Root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code


## 1. Divergent-selector controlled benchmark

This runs the disclosed oracle regimes, records estimated/oracle masks and regret, and explicitly identifies regimes in which a simple selector cannot be assumed equivalent to direct robust selection. It is not presented as real-data evidence.

In [2]:
if RUN_ALL and RUN_DIVERGENT_SELECTOR:
    settings=load_settings(CONFIG); logger=RunLogger(settings)
    try:
        result=run_regime_suite(settings,logger); logger.close(status='completed')
    except Exception:
        logger.close(status='failed'); raise
    summary=result.summary.copy()
    out=RESULTS/'divergent_selector_regimes.csv'; summary.to_csv(out,index=False)
    manifest={'scope':'controlled oracle regimes; selector divergence diagnostic, not real intervention evidence','source_run':str(result.run_dir),'regimes':summary['regime'].tolist(),'output':str(out)}
    (RESULTS/'divergent_selector_manifest.json').write_text(json.dumps(manifest,indent=2))
    display(summary)
else: print('Divergent selector study skipped.')

2026-08-18 10:53:47,780 | INFO | run_started | {"config_hash": "aef3e36943aafc81", "run_id": "curesim-quickstart-20260818T095347Z-08aef92f"}
2026-08-18 10:53:47,784 | INFO | planner_mode_resolved | {"base_feasible": true, "base_lower_improvement": 0.0, "cost": 0.0, "fatigue_upper": 0.1, "mode": "improvement", "provider_disparity_upper": 0.22, "relevance_delta_lower": 0.0}
2026-08-18 10:53:47,785 | INFO | portfolio_rejected | {"active_interventions": ["repeat_cap", "explore_slot", "tail_slot", "diversify", "novel_slot"], "coalition_mask": 31, "cost": 0.37, "fatigue_upper": 0.05, "lower_improvement": 0.23, "mode": "improvement", "provider_disparity_upper": 0.22, "relevance_delta_lower": -0.04999999999999999}
2026-08-18 10:53:47,786 | INFO | portfolio_rejected | {"active_interventions": ["explore_slot", "tail_slot", "diversify", "provider_balance"], "coalition_mask": 46, "cost": 0.36, "fatigue_upper": 0.1, "lower_improvement": 0.2, "mode": "improvement", "provider_disparity_upper": 0.22, 

,regime,description,horizon,expected_status,expected_estimated_selected,oracle_selected,observed_estimated_selected,observed_status,base_feasible,lower_improvement,estimated_selection_match,oracle_selection_match,estimated_jaccard,oracle_jaccard,oracle_regret
0,additive,Independent positive interventions with zero i...,12,improve_selected,repeat_cap;explore_slot;tail_slot;provider_bal...,repeat_cap;explore_slot;tail_slot;provider_bal...,repeat_cap;explore_slot;tail_slot;provider_bal...,improve_selected,True,0.21,True,True,1.0,1.0,0.00
1,complementary,Repeat cap and exploration are jointly valuabl...,12,improve_selected,repeat_cap;explore_slot,repeat_cap;explore_slot,repeat_cap;explore_slot,improve_selected,True,0.27,True,True,1.0,1.0,0.00
2,redundant,Long-tail and novelty overlap; one should be s...,12,improve_selected,tail_slot,tail_slot,tail_slot,improve_selected,True,0.15,True,True,1.0,1.0,0.00
3,antagonistic,Exploration and diversity are individually use...,12,improve_selected,explore_slot,explore_slot,explore_slot,improve_selected,True,0.14,True,True,1.0,1.0,0.00
4,delayed_fatigue_short,Repeat cap is costly at short horizon.,4,abstain_keep_base,,,,abstain_keep_base,True,-0.00,True,True,1.0,1.0,0.00
5,delayed_fatigue_long,Repeat cap yields delayed benefit at long hori...,12,improve_selected,repeat_cap,repeat_cap,repeat_cap,improve_selected,True,0.20,True,True,1.0,1.0,0.00
6,provider_repair_balancing,Infeasible base repaired most efficiently by p...,12,repair_selected,provider_balance,provider_balance,provider_balance,repair_selected,False,0.04,True,True,1.0,1.0,0.00
7,provider_repair_repeat,Infeasible base repaired most efficiently by r...,12,repair_selected,repeat_cap,repeat_cap,repeat_cap,repair_selected,False,0.10,True,True,1.0,1.0,0.00
8,misspecified_ambiguity,Estimated model favours repeat cap while true ...,12,improve_selected,repeat_cap,explore_slot,repeat_cap,improve_selected,True,0.16,True,False,1.0,0.0,0.19


## 2. Real/semi-real intervention validation gate

CURE-Rec policy selection cannot be validated on MovieLens ratings alone. This cell only runs if an audited intervention log or declared replay/world-model input is present.

In [3]:
if RUN_REAL_INTERVENTION:
    raise NotImplementedError('Provide audited logged slates, propensities, intervention assignment and outcomes before running this claim.')
else:
    print('Real/semi-real intervention validation blocked: no audited intervention log/world model is present.')

Real/semi-real intervention validation blocked: no audited intervention log/world model is present.


## 3. Integrated 8/10-player CURE scaling gate

The current 8/10 result is arithmetic attribution scaling. This cell prevents accidentally presenting it as integrated policy scaling until distinct operators are implemented in interventions.py, game.py and the simulator.

In [4]:
required={'session_length_cap','freshness_quota','provider_cooldown','category_coverage_quota'}
implemented=set()
if RUN_INTEGRATED_SCALING and not required.issubset(implemented):
    raise NotImplementedError(f'Missing integrated operators: {sorted(required-implemented)}')
else:
    print('Integrated 8/10 policy scaling blocked until distinct operators are implemented; arithmetic benchmark remains separately labelled.')

Integrated 8/10 policy scaling blocked until distinct operators are implemented; arithmetic benchmark remains separately labelled.


## 4. Completion manifest


In [5]:
manifest={'run_all':RUN_ALL,'divergent_selector':'executed_controlled_regimes' if RUN_DIVERGENT_SELECTOR else 'skipped','real_intervention':'blocked_no_audited_log','integrated_scaling':'blocked_missing_distinct_operators','yaml_changed':False,'claim_discipline':'blocked actions are not converted into claims'}
(RESULTS/'final_required_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))

{
  "run_all": true,
  "divergent_selector": "executed_controlled_regimes",
  "real_intervention": "blocked_no_audited_log",
  "integrated_scaling": "blocked_missing_distinct_operators",
  "yaml_changed": false,
  "claim_discipline": "blocked actions are not converted into claims"
}
